<a href="https://colab.research.google.com/github/areesha-del/AI-ML-Hands-on/blob/main/Home_Assignment_Advanced_Stock_Sequence_Models_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Home Assignment: Advanced Stock Forecasting (RNN, LSTM, GRU, BiLSTM, Attention)
Implement and compare:
- SimpleRNN
- LSTM
- GRU
- Bidirectional LSTM
- LSTM + Attention

**Evaluation:** RMSE + MAE, plots, and a final comparison table.

> Works in Google Colab.

In [ ]:
!pip -q install yfinance
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

import tensorflow as tf
from tensorflow.keras.layers import (
    Input, Dense, SimpleRNN, LSTM, GRU, Bidirectional,
    Attention, LayerNormalization, Flatten
)
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping

print('TensorFlow:', tf.__version__)

## Part A — Dataset

In [ ]:
TICKER = 'TSLA'
df = yf.download(TICKER, start='2017-01-01', progress=False)
print('Dataset shape:', df.shape)
display(df.head())

close = df['Close'].dropna().values.reshape(-1, 1)
plt.figure()
plt.plot(close)
plt.title(f'{TICKER} Close Price')
plt.xlabel('Days')
plt.ylabel('Price')
plt.show()

## Part B — Preprocessing

In [ ]:
scaler = MinMaxScaler()
close_scaled = scaler.fit_transform(close)

def make_sequences(series, lookback=30):
    X, y = [], []
    for i in range(lookback, len(series)):
        X.append(series[i-lookback:i])
        y.append(series[i])
    return np.array(X), np.array(y)

LOOKBACK = 30
X, y = make_sequences(close_scaled, LOOKBACK)

split = int(0.8 * len(X))
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

print('Train:', X_train.shape, y_train.shape)
print('Test :', X_test.shape, y_test.shape)

## Helper: Fit + Evaluate

In [ ]:
def fit_eval(model, name):
    model.compile(optimizer='adam', loss='mse')
    es = EarlyStopping(monitor='val_loss', patience=7, restore_best_weights=True)
    hist = model.fit(
        X_train, y_train,
        validation_split=0.2,
        epochs=80,
        batch_size=32,
        callbacks=[es],
        verbose=0
    )
    pred = model.predict(X_test, verbose=0)
    rmse = float(np.sqrt(mean_squared_error(y_test, pred)))
    mae  = float(mean_absolute_error(y_test, pred))

    plt.figure()
    plt.plot(hist.history['loss'], label='train')
    plt.plot(hist.history['val_loss'], label='val')
    plt.title(f'{name} Loss')
    plt.legend()
    plt.show()

    plt.figure()
    plt.plot(y_test[:200], label='Actual')
    plt.plot(pred[:200], label='Pred')
    plt.title(f'{name} Prediction (scaled)')
    plt.legend()
    plt.show()

    return rmse, mae

## Model 1 — SimpleRNN

In [ ]:
inp = Input(shape=(LOOKBACK, 1))
x = SimpleRNN(64)(inp)
out = Dense(1)(x)
model_rnn = Model(inp, out)
rnn_rmse, rnn_mae = fit_eval(model_rnn, 'SimpleRNN')

## Model 2 — LSTM

In [ ]:
inp = Input(shape=(LOOKBACK, 1))
x = LSTM(64)(inp)
out = Dense(1)(x)
model_lstm = Model(inp, out)
lstm_rmse, lstm_mae = fit_eval(model_lstm, 'LSTM')

## Model 3 — GRU

In [ ]:
inp = Input(shape=(LOOKBACK, 1))
x = GRU(64)(inp)
out = Dense(1)(x)
model_gru = Model(inp, out)
gru_rmse, gru_mae = fit_eval(model_gru, 'GRU')

## Model 4 — Bidirectional LSTM

In [ ]:
inp = Input(shape=(LOOKBACK, 1))
x = Bidirectional(LSTM(64))(inp)
out = Dense(1)(x)
model_bilstm = Model(inp, out)
bilstm_rmse, bilstm_mae = fit_eval(model_bilstm, 'BiLSTM')

## Model 5 — LSTM + Attention (Self-Attention)

In [ ]:
inp = Input(shape=(LOOKBACK, 1))
x = LSTM(64, return_sequences=True)(inp)
attn = Attention()([x, x])
x = LayerNormalization()(x + attn)
x = Flatten()(x)
x = Dense(64, activation='relu')(x)
out = Dense(1)(x)
model_attn = Model(inp, out)
attn_rmse, attn_mae = fit_eval(model_attn, 'LSTM + Attention')

## Final Comparison Table (Required)

In [ ]:
results = pd.DataFrame([
    {'Model':'SimpleRNN',        'RMSE':rnn_rmse,    'MAE':rnn_mae},
    {'Model':'LSTM',             'RMSE':lstm_rmse,   'MAE':lstm_mae},
    {'Model':'GRU',              'RMSE':gru_rmse,    'MAE':gru_mae},
    {'Model':'BiLSTM',           'RMSE':bilstm_rmse, 'MAE':bilstm_mae},
    {'Model':'LSTM + Attention', 'RMSE':attn_rmse,   'MAE':attn_mae},
])
display(results.sort_values('RMSE'))

## Conceptual Questions (Write Answers)
1) What is the vanishing gradient problem?
2) Why does LSTM handle long-term dependency better than SimpleRNN?
3) Difference between GRU and LSTM?
4) Why does Attention help in sequence modeling?

✅ Submit: notebook + 1-page PDF report.